In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.data_postprocessing import obtain_shoreline
from src.data_processing.dataset_loader import CoastData
from scipy.spatial import cKDTree

import pandas as pd

In [2]:
def get_coords(data_path, stations=[], get_mask_coords=False, for_matlab=False, output_folder="", is_oblique=True):
    data = CoastData(data_path)

    if 'global' not in stations:
        stations.append('global')

    filtered_data = data.get_images(get_all_metadata=True, get_mask=False) # All the data

    global_distance_all_points = {}
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for item in filtered_data:
        if get_mask_coords:
            coords_gt_u = item['metadata']["image"]["shoreline"]["coordinates"]['u']
            coords_gt_v = item['metadata']["image"]["shoreline"]["coordinates"]['v']
            if for_matlab:
                coords_gt_u = np.array(coords_gt_u) + 1
                coords_gt_v = np.array(coords_gt_v) + 1
            coords_gt = np.column_stack((coords_gt_u, coords_gt_v))

        coords_pred_u = item['metadata']['image']['predicted_shoreline']["coordinates"]['u']
        coords_pred_v = item['metadata']['image']['predicted_shoreline']["coordinates"]['v']
        if for_matlab:
            coords_pred_u = np.array(coords_pred_u) + 1
            coords_pred_v = np.array(coords_pred_v) + 1
        coords_pred = np.column_stack((coords_pred_u, coords_pred_v))


        export_path = item['metadata']['image']['oblique_path'] if is_oblique else item['metadata']['image']['filename']
        export_path = os.path.basename(export_path)
        export_path = export_path.replace("snap", "shoreline") if is_oblique else export_path.replace("image", "shoreline")
        export_path = export_path.replace(".jpg", ".csv")
        # print(f"Processing station: {station}, image: {export_path}")
        station_name = item['metadata']['image']['site']['CSname'] if len(stations) > 1 else 'global'

        new_stations = [station_name, 'global'] if len(stations) > 1 else ['global']

        for station in new_stations:
            if station not in global_distance_all_points:
                global_distance_all_points[station] = {
                    "oblique_path": [],
                    "coords": [],
                    "mask_coords": [],
                }
            global_distance_all_points[station]["oblique_path"].append(export_path)
            global_distance_all_points[station]["coords"].append(coords_pred)
            if get_mask_coords:
                global_distance_all_points[station]["mask_coords"].append(coords_gt)

        path_to_save = os.path.join(output_folder, export_path)
        df = pd.DataFrame(coords_pred, columns=['u', 'v'])
        df.to_csv(path_to_save, index=False)

    for station in stations:
        print(f"Station: {station}, Number of points: {len(global_distance_all_points[station]['coords'])}")


In [3]:
stations = ['global']
# stations = ['agrelo', 'arenaldentem', 'cadiz', 'cies', 'samarador']

## Experiment 1

### Oblique

In [4]:
# Oblique results
data_path_bilstm = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment1/oblique/BiLSTM/"))
output_path_bilstm = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment1/oblique/BiLSTM/"))

get_coords(data_path_bilstm, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_bilstm)

CoastData: global - 174 images
Station: global, Number of points: 174


## Rectified

In [5]:
# Oblique results
data_path_bilstm = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment1/rectified/BiLSTM/"))
output_path_bilstm = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment1/rectified/BiLSTM/"))

get_coords(data_path_bilstm, stations=stations, get_mask_coords=False, for_matlab=True, output_folder=output_path_bilstm, is_oblique=False)

CoastData: global - 174 images
Station: global, Number of points: 174


## Experiment 2
### Oblique

In [6]:
# Oblique results
data_path_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/oblique/UNet/"))
output_path_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment2/oblique/UNet/"))
data_path_attention_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/oblique/AttentionUNet/"))
output_path_attention_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment2/oblique/AttentionUNet/"))
data_path_deeplabv3 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/oblique/DeepLabV3/"))
output_path_deeplabv3 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment2/oblique/DeepLabV3/"))
data_path_ducknet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/oblique/DuckNet/"))
output_path_ducknet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment2/oblique/DuckNet/"))


get_coords(data_path_unet, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_unet)
get_coords(data_path_attention_unet, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_attention_unet)
get_coords(data_path_deeplabv3, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_deeplabv3)
get_coords(data_path_ducknet, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_ducknet)

CoastData: global - 174 images
Station: global, Number of points: 174
CoastData: global - 174 images
Station: global, Number of points: 174
CoastData: global - 174 images
Station: global, Number of points: 174
CoastData: global - 174 images
Station: global, Number of points: 174


## Rectified

In [7]:
# Rectified results
data_path_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/UNet/"))
output_path_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment2/rectified/UNet/"))
data_path_attention_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/AttentionUNet/"))
output_path_attention_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment2/rectified/AttentionUNet/"))
data_path_deeplabv3 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/DeepLabV3/"))
output_path_deeplabv3 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment2/rectified/DeepLabV3/"))
data_path_ducknet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/DuckNet/"))
output_path_ducknet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment2/rectified/DuckNet/"))


get_coords(data_path_unet, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_unet, is_oblique=False)
get_coords(data_path_attention_unet, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_attention_unet, is_oblique=False)
get_coords(data_path_deeplabv3, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_deeplabv3, is_oblique=False)
get_coords(data_path_ducknet, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_ducknet, is_oblique=False)

CoastData: global - 174 images
Station: global, Number of points: 174
CoastData: global - 174 images
Station: global, Number of points: 174
CoastData: global - 174 images
Station: global, Number of points: 174
CoastData: global - 174 images
Station: global, Number of points: 174


## Experiment 3
### Oblique

In [8]:
# Oblique results
data_path_deeplabv3_256x512 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment3/oblique/DeepLabV3_256x512/"))
output_path_deeplabv3_256x512 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment3/oblique/DeepLabV3_256x512/"))
data_path_deeplabv3_256x1024 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment3/oblique/DeepLabV3_256x1024/"))
output_path_deeplabv3_256x1024 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment3/oblique/DeepLabV3_256x1024/"))
data_path_deeplabv3_512x512 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment3/oblique/DeepLabV3_512x512/"))
output_path_deeplabv3_512x512 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment3/oblique/DeepLabV3_512x512/"))

get_coords(data_path_deeplabv3_256x512, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_deeplabv3_256x512)
get_coords(data_path_deeplabv3_256x1024, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_deeplabv3_256x1024)
get_coords(data_path_deeplabv3_512x512, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_deeplabv3_512x512)

CoastData: global - 174 images
Station: global, Number of points: 174
CoastData: global - 174 images
Station: global, Number of points: 174
CoastData: global - 174 images
Station: global, Number of points: 174


## Rectified

In [ ]:
# Rectified results
# data_path_deeplabv3_256x256 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment3/rectified/DeepLabV3_256x256/"))
# output_path_deeplabv3_256x256 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment3/rectified/DeepLabV3_256x256/"))
data_path_deeplabv3_256x512 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment3/rectified/DeepLabV3_256x512/"))
output_path_deeplabv3_256x512 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment3/rectified/DeepLabV3_256x512/"))
data_path_deeplabv3_256x1024 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment3/rectified/DeepLabV3_256x1024/"))
output_path_deeplabv3_256x1024 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment3/rectified/DeepLabV3_256x1024/"))
data_path_deeplabv3_512x512 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment3/rectified/DeepLabV3_512x512/"))
output_path_deeplabv3_512x512 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/csv/experiment3/rectified/DeepLabV3_512x512/"))

# get_coords(data_path_deeplabv3_256x256, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_deeplabv3_256x256, is_oblique=False)
get_coords(data_path_deeplabv3_256x512, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_deeplabv3_256x512, is_oblique=False)
get_coords(data_path_deeplabv3_256x1024, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_deeplabv3_256x1024, is_oblique=False)
get_coords(data_path_deeplabv3_512x512, stations=stations, get_mask_coords=True, for_matlab=True, output_folder=output_path_deeplabv3_512x512, is_oblique=False)

CoastData: global - 174 images
Station: global, Number of points: 174
CoastData: global - 174 images
Station: global, Number of points: 174
CoastData: global - 174 images
Station: global, Number of points: 174
CoastData: global - 174 images
Station: global, Number of points: 174
